In [ ]:
import sys
from pathlib import Path

# Add workspace root to path so we can import infraScan
workspace_root = Path("/cluster/home/lkuehner/MSc_Thesis/infraScan")
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

print(f"✓ Path setup: {sys.path[0]}")

In [ ]:
import matplotlib

if not hasattr(matplotlib.rcParams, "_get"):
    matplotlib.rcParams._get = matplotlib.rcParams.__getitem__


## ROAD

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

os.environ["USE_PYGEOS"] = "0"

# Repo-Root automatisch finden
workspace_root = Path(__file__).resolve().parents[2]
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from infraScan.infraScanRoad import settings
from infraScan.infraScanRoad.pipeline import (
    phase_1_initialization,
    phase_3_infrastructure_developments,
    phase_5_costs_and_accesibility,
    phase_6_travel_time_savings,
)

# Ganz wichtig auf Euler:
settings.MAIN = str(workspace_root)
os.chdir(settings.MAIN)

# Optional: falls du gezielt OD-TT laufen lassen willst
settings.travel_time_savings_method = "od"

runtimes = {}

limits_corridor, boundary_plot, innerboundary, outerboundary = phase_1_initialization(runtimes)
network, limits_variables, generated_points, current_points, current_access_points = phase_3_infrastructure_developments(
    innerboundary, outerboundary, runtimes
)
voronoi_tt = phase_5_costs_and_accesibility(limits_variables, runtimes)
phase_6_travel_time_savings(runtimes)

print("Done")
print(runtimes)


## Plot multimode random scenarios for selected scenario IDs

This cell builds the shared multimode scenario components and highlights the selected IDs in the representative-scenario plot. It does not materialize road or rail OD scenarios.


In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

os.environ["USE_PYGEOS"] = "0"
os.environ.setdefault("MPLBACKEND", "Agg")

# In a notebook __file__ is usually not defined, so use cwd and walk up to MSc_Thesis.
cwd = Path.cwd().resolve()
workspace_root = next((p for p in [cwd, *cwd.parents] if (p / "infraScan").exists() and (p / "data").exists()), cwd)
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))
os.chdir(workspace_root)


from infraScan.infraScanIntegrated import settings as integrated_settings
from infraScan.infraScanIntegrated.random_scenarios_multimode import (
    SCENARIO_PLOTS_DIR,
    build_shared_scenario_components,
    build_shared_scenario_summary,
    save_shared_scenario_components,
    save_shared_scenario_summary,
    save_representative_scenario_selection,
    plot_shared_scenario_components,
)

selected_scenario_names = [
    "scenario_29", "scenario_61", "scenario_66", "scenario_76", "scenario_83",
    "scenario_81", "scenario_71", "scenario_45", "scenario_31", "scenario_3",
    "scenario_58", "scenario_14", "scenario_50", "scenario_77", "scenario_41",
    "scenario_84", "scenario_53", "scenario_1", "scenario_87", "scenario_19",
]
selected_scenario_numbers = [int(name.split("_")[1]) for name in selected_scenario_names]

# Need to generate at least up to the largest selected scenario number.
num_scenarios = max(max(selected_scenario_numbers), int(integrated_settings.amount_of_scenarios))

components = build_shared_scenario_components(
    start_year=integrated_settings.start_year_scenario,
    end_year=integrated_settings.end_year_scenario,
    num_of_scenarios=num_scenarios,
)
summary_df = build_shared_scenario_summary(
    components,
    valuation_year=integrated_settings.start_valuation_year,
)

selected_df = summary_df[summary_df["scenario"].isin(selected_scenario_names)].copy()
selected_df["selection_order"] = selected_df["scenario"].map(
    {name: order for order, name in enumerate(selected_scenario_names, start=1)}
)
selected_df = selected_df.sort_values("selection_order")

# The scenario tables use 0-based scenario indices internally: scenario_1 -> 0.
selected_idx = [number - 1 for number in selected_scenario_numbers]

components_selected = components.copy()
for key in ["modal_split_road", "modal_split_rail", "modal_split_other", "distance_per_person"]:
    components_selected[key] = components[key][components[key]["scenario"].isin(selected_idx)].copy()

components_selected["population_scenarios"] = {
    district: df[df["scenario"].isin(selected_idx)].copy()
    for district, df in components["population_scenarios"].items()
}
components_selected["meta"] = dict(components["meta"])
components_selected["meta"]["num_of_scenarios"] = len(selected_idx)

summary_selected_df = summary_df[summary_df["scenario"].isin(selected_scenario_names)].copy()

output_dir = Path(SCENARIO_PLOTS_DIR) / "selected_20_only"
output_dir.mkdir(parents=True, exist_ok=True)

components_path = workspace_root / "data" / "infraScanIntegrated" / "scenarios" / "shared_components_selected_20_only.pkl"
summary_path = workspace_root / "data" / "infraScanIntegrated" / "scenarios" / "shared_scenario_summary_selected_20_only.csv"
selection_path = workspace_root / "data" / "infraScanIntegrated" / "scenarios" / "representative_scenario_selection_selected_20_only.csv"

save_shared_scenario_components(components_selected, str(components_path))
save_shared_scenario_summary(summary_selected_df, str(summary_path))
save_representative_scenario_selection(selected_df, str(selection_path))
plot_shared_scenario_components(components_selected, summary_selected_df, selected_df, str(output_dir))

print(f"Generated {num_scenarios} multimode scenarios, plotted only {len(selected_idx)} selected scenarios")
print(f"Selected scenarios: {selected_scenario_names}")
print(f"Components: {components_path}")
print(f"Summary: {summary_path}")
print(f"Selection: {selection_path}")
print(f"Plots: {output_dir}")
selected_df[["selection_order", "scenario", "shared_future_score", "road_modal_split", "rail_modal_split", "other_modal_split", "distance_per_person"]]


## Select 20 multimode scenarios with K-Means clustering

This cell selects 20 representative scenarios from the generated multimode scenario space using standardized K-Means features, then plots only those selected scenarios.


In [8]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

os.environ["USE_PYGEOS"] = "0"
os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-lkuehner")

cwd = Path.cwd().resolve()
workspace_root = next((p for p in [cwd, *cwd.parents] if (p / "infraScan").exists() and (p / "data").exists()), cwd)
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))
os.chdir(workspace_root)

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from infraScan.infraScanIntegrated import settings as integrated_settings
from infraScan.infraScanIntegrated.random_scenarios_multimode import (
    SCENARIO_PLOTS_DIR,
    build_shared_scenario_components,
    build_shared_scenario_summary,
    save_shared_scenario_components,
    save_shared_scenario_summary,
    save_representative_scenario_selection,
    plot_shared_scenario_components,
)

num_scenarios = int(integrated_settings.amount_of_scenarios)
n_clusters = 20

components = build_shared_scenario_components(
    start_year=integrated_settings.start_year_scenario,
    end_year=integrated_settings.end_year_scenario,
    num_of_scenarios=num_scenarios,
)
summary_df = build_shared_scenario_summary(
    components,
    valuation_year=integrated_settings.start_valuation_year,
)

feature_cols = [
    "population_growth_factor",
    "road_modal_split",
    "rail_modal_split",
    "other_modal_split",
    "distance_per_person",
    "road_demand_proxy",
    "rail_demand_proxy",
]

features = summary_df[feature_cols].astype(float).replace([np.inf, -np.inf], np.nan)
features = features.fillna(features.median(numeric_only=True))
X = StandardScaler().fit_transform(features)

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=50)
labels = kmeans.fit_predict(X)

selected_positions = []
for cluster_id in range(n_clusters):
    cluster_positions = np.flatnonzero(labels == cluster_id)
    center = kmeans.cluster_centers_[cluster_id]
    distances = np.linalg.norm(X[cluster_positions] - center, axis=1)
    selected_positions.append(cluster_positions[np.argmin(distances)])

selected_df = summary_df.iloc[selected_positions].copy()
selected_df["cluster"] = range(n_clusters)
selected_df = selected_df.sort_values("shared_future_score").reset_index(drop=True)
selected_df["selection_order"] = np.arange(1, len(selected_df) + 1)
selected_scenario_names = selected_df["scenario"].tolist()
selected_idx = selected_df["scenario_idx"].astype(int).tolist()

components_selected = components.copy()
for key in ["modal_split_road", "modal_split_rail", "modal_split_other", "distance_per_person"]:
    components_selected[key] = components[key][components[key]["scenario"].isin(selected_idx)].copy()

components_selected["population_scenarios"] = {
    district: df[df["scenario"].isin(selected_idx)].copy()
    for district, df in components["population_scenarios"].items()
}
components_selected["meta"] = dict(components["meta"])
components_selected["meta"]["num_of_scenarios"] = len(selected_idx)

summary_selected_df = summary_df[summary_df["scenario"].isin(selected_scenario_names)].copy()

output_dir = Path(SCENARIO_PLOTS_DIR) / "selected_20_kmeans"
output_dir.mkdir(parents=True, exist_ok=True)

components_path = workspace_root / "data" / "infraScanIntegrated" / "scenarios" / "shared_components_selected_20_kmeans.pkl"
summary_path = workspace_root / "data" / "infraScanIntegrated" / "scenarios" / "shared_scenario_summary_selected_20_kmeans.csv"
selection_path = workspace_root / "data" / "infraScanIntegrated" / "scenarios" / "representative_scenario_selection_selected_20_kmeans.csv"

save_shared_scenario_components(components_selected, str(components_path))
save_shared_scenario_summary(summary_selected_df, str(summary_path))
save_representative_scenario_selection(selected_df, str(selection_path))
plot_shared_scenario_components(components_selected, summary_selected_df, selected_df, str(output_dir))

print(f"Generated {num_scenarios} multimode scenarios")
print(f"Selected {len(selected_df)} K-Means representative scenarios")
print(f"Selected scenarios: {selected_scenario_names}")
print(f"Components: {components_path}")
print(f"Summary: {summary_path}")
print(f"Selection: {selection_path}")
print(f"Plots: {output_dir}")
selected_df[["selection_order", "cluster", "scenario", "shared_future_score", *feature_cols]]


Generated 100 multimode scenarios
Selected 20 K-Means representative scenarios
Selected scenarios: ['scenario_23', 'scenario_47', 'scenario_26', 'scenario_85', 'scenario_100', 'scenario_39', 'scenario_81', 'scenario_11', 'scenario_55', 'scenario_31', 'scenario_96', 'scenario_38', 'scenario_41', 'scenario_84', 'scenario_36', 'scenario_35', 'scenario_63', 'scenario_98', 'scenario_97', 'scenario_10']
Components: /cluster/home/lkuehner/MSc_Thesis/data/infraScanIntegrated/scenarios/shared_components_selected_20_kmeans.pkl
Summary: /cluster/home/lkuehner/MSc_Thesis/data/infraScanIntegrated/scenarios/shared_scenario_summary_selected_20_kmeans.csv
Selection: /cluster/home/lkuehner/MSc_Thesis/data/infraScanIntegrated/scenarios/representative_scenario_selection_selected_20_kmeans.csv
Plots: /cluster/home/lkuehner/MSc_Thesis/plots/scenarios/selected_20_kmeans


,selection_order,cluster,scenario,shared_future_score,population_growth_factor,road_modal_split,rail_modal_split,other_modal_split,distance_per_person,road_demand_proxy,rail_demand_proxy
0,1,4,scenario_23,0.234286,1.150329,0.730852,0.205416,0.063733,30.654008,3.909007e+07,1.098679e+07
1,2,6,scenario_47,0.240000,1.091031,0.696770,0.233123,0.070107,31.206803,3.598352e+07,1.203924e+07
2,3,11,scenario_26,0.258571,0.966343,0.673079,0.236587,0.090335,28.979772,2.859037e+07,1.004949e+07
3,4,0,scenario_85,0.277143,1.144823,0.652774,0.271414,0.075812,31.663512,3.589121e+07,1.492305e+07
4,5,12,scenario_100,0.380000,1.200298,0.682894,0.205157,0.111949,33.639846,4.182386e+07,1.256484e+07
5,6,19,scenario_39,0.411429,1.226531,0.678266,0.248152,0.073582,35.560023,4.487128e+07,1.641671e+07
6,7,16,scenario_81,0.431429,1.233428,0.616166,0.286282,0.097551,32.830550,3.784580e+07,1.758387e+07
7,8,13,scenario_11,0.455714,1.324747,0.696864,0.224007,0.079128,35.038470,4.906299e+07,1.577132e+07
8,9,15,scenario_55,0.485714,1.277797,0.660618,0.246136,0.093246,35.868868,4.592591e+07,1.711132e+07
9,10,9,scenario_31,0.490000,1.278750,0.652746,0.267781,0.079473,36.810905,4.660513e+07,1.911919e+07
